# Stage 2: Comprehensive Merged Data Exploratory Analysis

Taking place post-merge and before preprocessing analysis with the purpose of understanding distributions, validating quality, identifying issues

## 1. Essentials

### 1.1 Setup & Imports

In [ ]:
from pathlib import Path
import time

from IPython.display import display
from matplotlib.gridspec import GridSpec
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import kruskal, kurtosis, skew, zscore
import seaborn as sns

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("husl")
sns.set_context("notebook", font_scale=1.1)
%matplotlib inline

plt.rcParams["figure.dpi"] = 100
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["font.size"] = 10


print("All imports successful")

### 1.2 Config

In [ ]:
# Paths
DATA_PATH = Path("../data/raw/wesad/WESAD/merged")
FIGURES_PATH = Path("../results/figures")
REPORTS_PATH = Path("../results/eda_reports")

# Create directories if not exits
FIGURES_PATH.mkdir(parents=True, exist_ok=True)
REPORTS_PATH.mkdir(parents=True, exist_ok=True)

# Dataset configuration
SUBJECT_IDS = [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 16, 17]

# Sampling rates (Hz) from WESAD documentation
SF_CHEST = 700
SF_BVP = 64
SF_EDA = 4
SF_TEMP = 4
SF_ACC_WRIST = 32

# Label definitions (from WESAD README)
LABEL_NAMES = {
    0: "Not defined / Transient",
    1: "Baseline",
    2: "Stress",
    3: "Amusement",
    4: "Meditation",
    5: "Ignore (per README)",
    6: "Ignore (per README)",
    7: "Ignore (per README)",
}

# Usable labels for stress detection
USABLE_LABELS = [1, 2, 3]
LABEL_SHORT = {1: "Baseline", 2: "Stress", 3: "Amusement"}

# Color palette for visualization
LABEL_COLORS = {
    1: "#2ecc71",  # Green Baseline
    2: "#e74c3c",  # Red Stress
    3: "#3498db",  # BlueAmusement
}

STRING_LABEL_COLORS = {
    "Baseline": "#2ecc71",
    "Stress": "#e74c3c",
    "Amusement": "#3498db",
}

print("  Configuration loaded")
print(f"  Data path: {DATA_PATH.absolute()}")
print(f"  Figures: {FIGURES_PATH.absolute()}")
print(f"  Reports: {REPORTS_PATH.absolute()}")
print(f"  Subjects: {len(SUBJECT_IDS)}")

### 1.3 Load Merged Data

In [ ]:
start_time = time.time()

print("LOADING MERGED DATA...")

# Load chest data
chest_path = DATA_PATH / "merged_chest.pkl"
print(f"\nChest: {chest_path}")
chest_df = pd.read_pickle(chest_path)
print(f"  Shape: {chest_df.shape}")
print(f"  Columns: {list(chest_df.columns)}")

# Load wrist data
wrist_path = DATA_PATH / "merged_wrist.pkl"
print(f"\nWrist: {wrist_path}")
wrist_df = pd.read_pickle(wrist_path)
print(f"  Shape: {wrist_df.shape}")
print(f"  Columns: {list(wrist_df.columns)}")

# memory use
chest_mem_mb = chest_df.memory_usage(deep=True).sum() / 1024**2
wrist_mem_mb = wrist_df.memory_usage(deep=True).sum() / 1024**2

print("\n")
print("MEMORY USAGE")
print(f"  Chest: {chest_mem_mb:.2f} MB")
print(f"  Wrist: {wrist_mem_mb:.2f} MB")
print(f"  Total: {chest_mem_mb + wrist_mem_mb:.2f} MB")

load_time = time.time() - start_time
print(f"\n Data loaded in {load_time:.1f} seconds")

### 1.4 Data Preview

In [ ]:
print("CHEST DATA - First 10 rows:")
display(chest_df.head(10))

In [ ]:
print("WRIST DATA - First 10 rows:")
display(wrist_df.head(10))

### 1.5 General Info

In [ ]:
print("CHEST DATA INFO:")
chest_df.info()

In [ ]:
print("WRIST DATA INFO:")
wrist_df.info()

In [ ]:
print("CHEST DATA DESCRIPTIVE STATISTICS:")
display(chest_df.describe().T)

In [ ]:
print("WRIST DATA DESCRIPTIVE STATISTICS:")
display(wrist_df.describe().T)

# 2. Label Distribution

### 2.1 Overall Label Distribution (Includes All Labels)

In [ ]:
print("LABEL DISTRIBUTION (ALL LABELS)")

# Chest
chest_label_counts = chest_df["label"].value_counts().sort_index()
print("\nChest:")
for label, count in chest_label_counts.items():
    pct = (count / len(chest_df)) * 100
    label_name = LABEL_NAMES.get(int(label), "Unknown")
    print(f"  {int(label)}: {label_name:30s} - {count:10,} samples ({pct:5.2f}%)")

# Wrist labels
wrist_label_counts = wrist_df["label"].value_counts().sort_index()
print("\nWrist:")
for label, count in wrist_label_counts.items():
    pct = (count / len(wrist_df)) * 100
    label_name = LABEL_NAMES.get(int(label), "Unknown")
    print(f"  {int(label)}: {label_name:30s} - {count:10,} samples ({pct:5.2f}%)")

### 2.2 Usable Labels Distribution (1, 2, 3 Baseline Stress Amusement)

In [ ]:
chest_usable = chest_df[chest_df["label"].isin(USABLE_LABELS)]
wrist_usable = wrist_df[wrist_df["label"].isin(USABLE_LABELS)]

print("USABLE LABELS (Baseline, Stress, Amusement)")

print(
    f"\nChest: {len(chest_usable):,} / {len(chest_df):,} samples ({len(chest_usable)/len(chest_df)*100:.1f}%)"
)
for label in USABLE_LABELS:
    count = (chest_usable["label"] == label).sum()
    pct = (count / len(chest_usable)) * 100
    print(f"  {LABEL_SHORT[label]:10s}: {count:10,} ({pct:5.2f}%)")

print(
    f"\nWrist: {len(wrist_usable):,} / {len(wrist_df):,} samples ({len(wrist_usable)/len(wrist_df)*100:.1f}%)"
)
for label in USABLE_LABELS:
    count = (wrist_usable["label"] == label).sum()
    pct = (count / len(wrist_usable)) * 100
    print(f"  {LABEL_SHORT[label]:10s}: {count:10,} ({pct:5.2f}%)")

### 2.3 Pie Charts Visualization for Label Distribution: 

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

# Chest pie chart
chest_counts = chest_df["label"].value_counts().sort_index()
colors = [
    "#95a5a6" if lbl not in USABLE_LABELS else LABEL_COLORS.get(lbl, "#95a5a6")
    for lbl in chest_counts.index
]

wedges, texts, autotexts = ax1.pie(
    chest_counts,
    colors=colors,
    autopct="%1.1f%%",
    startangle=90,
    textprops={"fontsize": 11, "weight": "bold"},
)

# White text on slices for readability
for autotext in autotexts:
    autotext.set_color("white")

# Legend with full details
legend_labels = [
    f"{LABEL_NAMES.get(int(lbl), 'Unknown')}: {c:,} ({c/chest_counts.sum()*100:.1f}%)"
    for lbl, c in zip(chest_counts.index, chest_counts.values)
]
ax1.legend(
    legend_labels, loc="upper left", bbox_to_anchor=(1.05, 1), frameon=True, fontsize=9
)
ax1.set_title("Chest Data - All Labels", fontsize=14, fontweight="bold")

# Wrist pie char
wrist_counts = wrist_df["label"].value_counts().sort_index()
colors = [
    "#95a5a6" if lbl not in USABLE_LABELS else LABEL_COLORS.get(lbl, "#95a5a6")
    for lbl in wrist_counts.index
]

wedges, texts, autotexts = ax2.pie(
    wrist_counts,
    colors=colors,
    autopct="%1.1f%%",
    startangle=90,
    textprops={"fontsize": 11, "weight": "bold"},
)

for autotext in autotexts:
    autotext.set_color("white")

legend_labels = [
    f"{LABEL_NAMES.get(int(lbl), 'Unknown')}: {c:,} ({c/wrist_counts.sum()*100:.1f}%)"
    for lbl, c in zip(wrist_counts.index, wrist_counts.values)
]
ax2.legend(
    legend_labels, loc="upper left", bbox_to_anchor=(1.05, 1), frameon=True, fontsize=9
)
ax2.set_title("Wrist Data - All Labels", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.savefig(
    FIGURES_PATH / "02_label_distribution_all.png", dpi=300, bbox_inches="tight"
)
plt.show()

print("Saved: 02_label_distribution_all.png")

### 2.4 Bar Charts for Usable Labels Only

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Chest bar chart
chest_usable_counts = chest_usable["label"].value_counts().sort_index()
colors_chest = [LABEL_COLORS[lbl] for lbl in chest_usable_counts.index]
bars1 = ax1.bar(
    range(len(chest_usable_counts)),
    chest_usable_counts.values,
    color=colors_chest,
    alpha=0.8,
    edgecolor="black",
    linewidth=1.5,
)
ax1.set_xticks(range(len(chest_usable_counts)))
ax1.set_xticklabels([LABEL_SHORT[lbl] for lbl in chest_usable_counts.index])
ax1.set_ylabel("Sample Count", fontsize=12)
ax1.set_title("Chest Data - Usable Labels", fontsize=14, fontweight="bold")
ax1.grid(axis="y", alpha=0.3)

# value labels on bars
for bar in bars1:
    height = bar.get_height()
    ax1.text(
        bar.get_x() + bar.get_width() / 2.0,
        height,
        f"{int(height):,}",
        ha="center",
        va="bottom",
        fontsize=10,
        fontweight="bold",
    )

# Wrist bar chart
wrist_usable_counts = wrist_usable["label"].value_counts().sort_index()
colors_wrist = [LABEL_COLORS[lbl] for lbl in wrist_usable_counts.index]
bars2 = ax2.bar(
    range(len(wrist_usable_counts)),
    wrist_usable_counts.values,
    color=colors_wrist,
    alpha=0.8,
    edgecolor="black",
    linewidth=1.5,
)
ax2.set_xticks(range(len(wrist_usable_counts)))
ax2.set_xticklabels([LABEL_SHORT[lbl] for lbl in wrist_usable_counts.index])
ax2.set_ylabel("Sample Count", fontsize=12)
ax2.set_title("Wrist Data - Usable Labels", fontsize=14, fontweight="bold")
ax2.grid(axis="y", alpha=0.3)

# value labels on bars
for bar in bars2:
    height = bar.get_height()
    ax2.text(
        bar.get_x() + bar.get_width() / 2.0,
        height,
        f"{int(height):,}",
        ha="center",
        va="bottom",
        fontsize=10,
        fontweight="bold",
    )

plt.tight_layout()
plt.savefig(
    FIGURES_PATH / "02_label_distribution_usable.png", dpi=300, bbox_inches="tight"
)
plt.show()

print("Saved: 02_label_distribution_usable.png")

### 2.5 Per-Subject Label Distribution

In [ ]:
# Calculate label distribution per subject
subject_label_dist = {}

for sid in SUBJECT_IDS:
    chest_subj = chest_df[chest_df["sid"] == sid]
    label_counts = chest_subj["label"].value_counts()
    subject_label_dist[sid] = label_counts

# Create DataFrame for easier visualization
dist_df = pd.DataFrame(subject_label_dist).T.fillna(0).astype(int)
dist_df = dist_df[
    [float(lbl) for lbl in USABLE_LABELS if float(lbl) in dist_df.columns]
]
dist_df.columns = [LABEL_SHORT[int(c)] for c in dist_df.columns]

print("LABEL DISTRIBUTION PER SUBJECT (Usable Labels Only)")
display(dist_df)

# Plot
fig, ax = plt.subplots(figsize=(14, 6))
dist_df.plot(
    kind="bar",
    ax=ax,
    color=[LABEL_COLORS[lbl] for lbl in USABLE_LABELS],
    alpha=0.8,
    edgecolor="black",
    linewidth=1,
)
ax.set_xlabel("Subject ID", fontsize=12)
ax.set_ylabel("Sample Count", fontsize=12)
ax.set_title("Label Distribution per Subject", fontsize=14, fontweight="bold")
ax.legend(title="Label", fontsize=10)
ax.grid(axis="y", alpha=0.3)
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(FIGURES_PATH / "02_labels_per_subject.png", dpi=300, bbox_inches="tight")
plt.show()

print("Saved: 02_labels_per_subject.png")

## 2.6 - Crucial Question: are the differences in the individual distributions statistically significant?

According to the dataset readme, the same experimental protocol was implemented across all subjects. Fixed time periods were used for each condition, in a controlled lab environment and
with the same sampling rate. Therefore we expect the small differences observed to be due to minor timing variations or slightly different recording lengths. Any significant difference would
be a red flag for inconsistent protocol and data quality issues

In [ ]:
print("\nPurpose: Verify that all subjects followed the same experimental protocol")
print("Expected: NO significant differences in label proportions")

# ----------------------------------------------------------------------------
# Calculate proportions per subject
# ----------------------------------------------------------------------------
print("\n1. LABEL PROPORTIONS PER SUBJECT")

# Convert to proportions (each subject sums to 1.0)
proportions_df = dist_df.div(dist_df.sum(axis=1), axis=0) * 100  # As percentages

print("\nLabel proportions (%) per subject:")
display(proportions_df.round(2))

# ----------------------------------------------------------------------------
# Chi-Square Test
# ----------------------------------------------------------------------------
print("\n2. CHI-SQUARE TEST")
print("H0: Label proportions are the same across subjects")
print("H1: Label proportions differ significantly across subjects")

contingency_table = dist_df.values  # Subjects × Labels

chi2, p_value, dof, expected = stats.chi2_contingency(contingency_table)

print(f"\nChi-square statistic: {chi2:.2f}")
print(f"Degrees of freedom: {dof}")
print(f"P-value: {p_value:.6f}")

# Calculate effect size (Cramér's V)
n = contingency_table.sum()
min_dim = min(contingency_table.shape[0] - 1, contingency_table.shape[1] - 1)
cramers_v = np.sqrt(chi2 / (n * min_dim))
print(f"Cramér's V (effect size): {cramers_v:.4f}")

print("\nEffect size interpretation:")
print("  < 0.1: Negligible")
print("  0.1-0.3: Small")
print("  0.3-0.5: Medium")
print("  > 0.5: Large")

# The key insight: chi-square can be significant even with tiny differences
# when sample sizes are huge so we must check effect size

if p_value < 0.05:
    if cramers_v < 0.1:
        print("\n   Statistically significant but negligivle effect")
        print("   → Chi-square is significant due to large sample size")
        print("   → Effect size is negligible (Cramér's V < 0.1)")
        print("   → Actual differences are tiny and practically meaningless")
        print("   → Protocol was effectively consistent")
    elif cramers_v < 0.3:
        print("\n   Significant with small effect")
        print("   → Minor differences in proportions exist")
        print("   → Check CV and proportions to assess practical significance")
    else:
        print("\n   Significant with large effect")
        print("   → Protocol may not have been followed consistently")
        print("   → Investigate deviations")
else:
    print("\n   no significant differences")
    print("   → Protocol was followed consistently")

# ----------------------------------------------------------------------------
# Coefficient of Variation
# ----------------------------------------------------------------------------
print("\n\n2. COEFFICIENT OF VARIATION (CV)")
print("Measure relative variability in sample counts")

kw_results = {}
for label in USABLE_LABELS:
    label_name = LABEL_SHORT[label]
    label_counts = [
        chest_df[(chest_df["sid"] == sid) & (chest_df["label"] == label)].shape[0]
        for sid in SUBJECT_IDS
    ]

    kw_results[label_name] = {
        "Mean": np.mean(label_counts),
        "Std": np.std(label_counts),
        "Min": np.min(label_counts),
        "Max": np.max(label_counts),
        "CV": (
            (np.std(label_counts) / np.mean(label_counts) * 100)
            if np.mean(label_counts) > 0
            else 0
        ),
    }

print("\nPer-label statistics across subjects:")
print(f"{'Label':<12} {'Mean':<12} {'Std':<12} {'CV (%)':<10} {'Interpretation'}")

for label_name, stats_dict in kw_results.items():
    cv = stats_dict["CV"]

    # For controlled experiment, we expect CV < 10%
    if cv < 5:
        interpretation = "Excellent consistency "
    elif cv < 10:
        interpretation = "Good consistency "
    elif cv < 20:
        interpretation = "Acceptable variation"
    else:
        interpretation = "High variation  "

    print(
        f"{label_name:<12} {stats_dict['Mean']:<12.0f} {stats_dict['Std']:<12.1f} "
        f"{cv:<10.1f} {interpretation}"
    )

# ----------------------------------------------------------------------------
# Duration Consistency Analysis
# ----------------------------------------------------------------------------
print("\n\n3. EXPECTED vs ACTUAL DURATIONS")
print("Converting sample counts to approximate duration (minutes)")

for label in USABLE_LABELS:
    label_name = LABEL_SHORT[label]
    label_counts = [
        chest_df[(chest_df["sid"] == sid) & (chest_df["label"] == label)].shape[0]
        for sid in SUBJECT_IDS
    ]

    # Convert to minutes (700 Hz sampling rate)
    durations_minutes = [count / (SF_CHEST * 60) for count in label_counts]

    print(f"\n{label_name}:")
    print(f"  Mean duration: {np.mean(durations_minutes):.1f} minutes")
    print(f"  Std deviation: {np.std(durations_minutes):.1f} minutes")
    print(
        f"  Range: {np.min(durations_minutes):.1f} - {np.max(durations_minutes):.1f} minutes"
    )
    print(
        f"  CV: {(np.std(durations_minutes) / np.mean(durations_minutes) * 100):.1f}%"
    )

# ----------------------------------------------------------------------------
# ANOVA on Proportions
# ----------------------------------------------------------------------------
print("\n\n3. ONE-WAY ANOVA ON PROPORTIONS")
print("Tests if mean proportion differs across labels")

for label_idx, label in enumerate(USABLE_LABELS):
    label_name = LABEL_SHORT[label]

    # Get proportions for this label across all subjects
    proportions = proportions_df.iloc[:, label_idx].values

    # Test if variance is significant
    mean_prop = np.mean(proportions)
    std_prop = np.std(proportions)
    cv_prop = (std_prop / mean_prop * 100) if mean_prop > 0 else 0

    print(f"\n{label_name}:")
    print(f"  Mean proportion: {mean_prop:.2f}%")
    print(f"  Std deviation: {std_prop:.2f}%")
    print(f"  CV: {cv_prop:.2f}%")
    print(f"  Range: {proportions.min():.2f}% - {proportions.max():.2f}%")

    if cv_prop < 5:
        print("  → Excellent consistency")
    elif cv_prop < 10:
        print("  → Good consistency")
    else:
        print("  → Some variation")

# ----------------------------------------------------------------------------
# Visualize Proportions
# ----------------------------------------------------------------------------
print("\n\nGenerating proportion consistency plot...")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Stacked bar chart of proportions
proportions_df.plot(
    kind="bar",
    stacked=True,
    ax=axes[0],
    color=[LABEL_COLORS[lbl] for lbl in USABLE_LABELS],
    alpha=0.8,
    edgecolor="black",
    linewidth=1,
)
axes[0].set_xlabel("Subject ID", fontsize=12)
axes[0].set_ylabel("Proportion (%)", fontsize=12)
axes[0].set_title(
    "Label Proportions per Subject\n(Should be similar if protocol was consistent)",
    fontsize=12,
    fontweight="bold",
)
axes[0].legend(title="Label", bbox_to_anchor=(1.05, 1), loc="upper left")
axes[0].set_xticklabels([f"S{sid}" for sid in SUBJECT_IDS], rotation=45)
axes[0].grid(axis="y", alpha=0.3)

# Plot 2: Box plot of proportions
box_data = [proportions_df.iloc[:, i].values for i in range(len(USABLE_LABELS))]
box_labels = [LABEL_SHORT[lbl] for lbl in USABLE_LABELS]

bp = axes[1].boxplot(
    box_data,
    labels=box_labels,
    patch_artist=True,
    widths=0.6,
    showmeans=True,
    meanline=True,
)

for patch, label in zip(bp["boxes"], USABLE_LABELS):
    patch.set_facecolor(LABEL_COLORS[label])
    patch.set_alpha(0.7)

axes[1].set_xlabel("Label", fontsize=12)
axes[1].set_ylabel("Proportion (%) Across Subjects", fontsize=12)
axes[1].set_title(
    "Distribution of Label Proportions\n(Narrow boxes = consistent protocol)",
    fontsize=12,
    fontweight="bold",
)
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_PATH / "02_protocol_consistency.png", dpi=300, bbox_inches="tight")
plt.show()

print("Saved: 02_protocol_consistency.png")

# ----------------------------------------------------------------------------
# Final Verdict
# ----------------------------------------------------------------------------

# Check both CV and Cramér's V
all_cv_good = all(
    (np.std(proportions_df.iloc[:, i]) / np.mean(proportions_df.iloc[:, i]) * 100) < 10
    for i in range(len(USABLE_LABELS))
)

if cramers_v < 0.1 or all_cv_good:
    print("\nPROTOCOL CONSISTENCY CONFIRMED")
    print("\n   Evidence:")
    print("   • CV < 5% for all labels (excellent consistency)")
    print(f"   • Cramér's V = {cramers_v:.4f} (negligible effect size)")
    print("   • Duration variations < 1 minute per condition")

    print("\n   Interpretation:")
    print("   • Standardized protocol was followed correctly")
    print("   • All subjects underwent comparable experimental conditions")
    print("   • Chi-square significance is due to large sample size")
    print("   • Actual practical differences are negligible")

else:
    print("\n   PROTOCOL INCONSISTENCY DETECTED")
    print(f"\n   • Cramér's V = {cramers_v:.4f} (meaningful effect)")
    print("   • Some labels show CV > 10%")
    print("\n   Action needed: Investigate subject-specific deviations")

Conclusion:While chi-square test showed statistical significance (χ² = 9064.99, p < 0.001), effect size was negligible (Cramér's V = 0.0140),
with coefficient of variation < 5% for all conditions. This confirms excellent protocol consistency, and observed differences 
can be traced back to minor timing differences owing to manual experiment handling rather than systematic deviations.

The above findings confirm that Label distributions are statistically similar across subjects, meaning all subjects underwent comparable experiment 
conditions and data quality is high and suitable for analysis. Therefore, any potential 
differences in model performance will be due to true individual differences, not experimental artifacts. This also means LOSOCV 
in this case can provide a fair evaluation because subjects are comparable.





## 3: Data Quality Analysis

In [ ]:
print("MISSING DATA ANALYSIS")

# Check for NaN
chest_nans = chest_df.isnull().sum()
wrist_nans = wrist_df.isnull().sum()

print("\nChest Data:")
if chest_nans.sum() == 0:
    print(" No missing values")
else:
    print(f"  Total NaNs: {chest_nans.sum():,}")
    print("\n  NaN counts per column:")
    for col, count in chest_nans[chest_nans > 0].items():
        pct = (count / len(chest_df)) * 100
        print(f"    {col:15s}: {count:10,} ({pct:5.2f}%)")

print("\nWrist Data:")
if wrist_nans.sum() == 0:
    print(" No missing values")
else:
    print(f"  Total NaNs: {wrist_nans.sum():,}")
    print("\n  NaN counts per column:")
    for col, count in wrist_nans[wrist_nans > 0].items():
        pct = (count / len(wrist_df)) * 100
        print(f"    {col:15s}: {count:10,} ({pct:5.2f}%)")

### 3.2 Missing Data Heatmap

In [ ]:
# Create missing data heatmap
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Chest missing data (sample 10000 rows for visualization)
sample_size = min(10000, len(chest_df))
chest_sample = chest_df.sample(sample_size, random_state=42)
sns.heatmap(
    chest_sample.isnull(), yticklabels=False, cbar=True, cmap="RdYlGn_r", ax=ax1
)
ax1.set_title(
    f"Chest Data - Missing Values\n(Sample of {sample_size:,} rows)",
    fontsize=12,
    fontweight="bold",
)
ax1.set_xlabel("Columns", fontsize=10)

# Wrist missing data
wrist_sample = wrist_df.sample(sample_size, random_state=42)
sns.heatmap(
    wrist_sample.isnull(), yticklabels=False, cbar=True, cmap="RdYlGn_r", ax=ax2
)
ax2.set_title(
    f"Wrist Data - Missing Values\n(Sample of {sample_size:,} rows)",
    fontsize=12,
    fontweight="bold",
)
ax2.set_xlabel("Columns", fontsize=10)

plt.tight_layout()
plt.savefig(FIGURES_PATH / "03_missing_data_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()

print("Saved: 03_missing_data_heatmap.png")

### 3.3 Invalid Temperature Values
(expected to find none after incorporating interpolation in the resample_merge preprocessor)

In [ ]:
print("TEMPERATURE VALIDATION")

# Check chest temperature
chest_temp_col = "c_temp" if "c_temp" in chest_df.columns else "temp"
chest_invalid = (chest_df[chest_temp_col] <= 0).sum()

print(f"\nChest Temperature ({chest_temp_col}):")
print(f"  Invalid (≤0): {chest_invalid:,} samples")
if chest_invalid > 0:
    print(f"  WARNING: Still have {chest_invalid} invalid values")
    print("  This suggests interpolation didn't work or wasn't applied.")
else:
    print(" No invalid temperatures.")

print(f"  Min: {chest_df[chest_temp_col].min():.2f}°C")
print(f"  Max: {chest_df[chest_temp_col].max():.2f}°C")
print(f"  Mean: {chest_df[chest_temp_col].mean():.2f}°C")

# Check wrist temperature
wrist_temp_col = "w_temp"
wrist_invalid = (wrist_df[wrist_temp_col] <= 0).sum()

print(f"\nWrist Temperature ({wrist_temp_col}):")
print(f"  Invalid (≤0): {wrist_invalid:,} samples")
if wrist_invalid > 0:
    print(f"  WARNING: Still have {wrist_invalid} invalid values")
else:
    print(" No invalid temperatures")

print(f"  Min: {wrist_df[wrist_temp_col].min():.2f}°C")
print(f"  Max: {wrist_df[wrist_temp_col].max():.2f}°C")
print(f"  Mean: {wrist_df[wrist_temp_col].mean():.2f}°C")

if chest_invalid == 0 and wrist_invalid == 0:
    print("\nTemperature interpolation successful!")
else:
    print("ACTION NEEDED: Fix temperature interpolation in preprocessing code")

### 3.4 Negative Values Check


In [ ]:
print("NEGATIVE VALUE CHECK (All Sensors)")

# Check chest sensors for negative values
print("\nChest Data:")
for col in chest_df.columns:
    if col not in ["sid", "label"]:
        negative_count = (chest_df[col] < 0).sum()
        if negative_count > 0:
            pct = (negative_count / len(chest_df)) * 100
            print(f"  {col:15s}: {negative_count:8,} negative values ({pct:.3f}%)")

# Check wrist sensors for negative values
print("\nWrist Data:")
for col in wrist_df.columns:
    if col not in ["sid", "label"]:
        negative_count = (wrist_df[col] < 0).sum()
        if negative_count > 0:
            pct = (negative_count / len(wrist_df)) * 100
            print(f"  {col:15s}: {negative_count:8,} negative values ({pct:.3f}%)")


Note: Negative values here are normal (accelerometer can be negative when motion happens in negative axis direction, ecg measures electrical potential difference, also centered around
zero, similar with emg. Respiration and bvp are also centered around 0 so negative values there are also expected.)  The only category where negative values indicate erroneous artifacts 
is the wrist electrodermal activity as well as chest and wrist temperatures.

More specifically, we utilize domain expertise to verify which negative values are normal and how to handle the bad artefacts. 
- Accelerometers measure proper acceleration relative to free-fall. A negative value simply indicates acceleration in the opposite direction along that axis, therefore negative values are expected.
- Biopotentials (ECG, EMG) measure the differential electrical potential between two electrodes. The signal naturally oscillates positive and negative around a baseline (isoelectric line). With the mean neing around zero, the negative values here are also naturally occuring.
- BVP (Photoplethysmography): The BVP signal from the Empatica E4 is similarly filtered and centered in a way that creates negative valleys.
- Respiration measures chest expansion/contraction. It is a wave oscillating around a relative zero point (inhalation vs. exhalation), therefore negative values are acceptable.

- Skin conductance (EDA) is a physical property that cannot be negative (Conductance=1/Resistance>0), therefore these are bad artefacts (noise, or sensor errors). For 11 values out of 6 million, MCAR test will not offer any particular actionable insight as this is not a statistically significant amount by any means (0.000%). To avoid jitter and affecting window features such as eda_mean we perform linear interpolation.

### 3.5 Duplicate Rows Check

In [ ]:
print("DUPLICATE ROWS CHECK")

chest_dupes = chest_df.duplicated().sum()
wrist_dupes = wrist_df.duplicated().sum()

print(f"\nChest: {chest_dupes:,} duplicate rows")
print(f"Wrist: {wrist_dupes:,} duplicate rows")

if chest_dupes == 0 and wrist_dupes == 0:
    print("\n   No duplicate rows found!")
else:
    print("\n   Duplicates detected")

## 3.6 Physiological range check (implausibly high values)

In [ ]:
# Temperature Check (Values outside the 25 - 40 range should be investigated)
print(f"Max Chest Temp: {chest_df['c_temp'].max():.2f} °C")
print(f"Max Wrist Temp: {wrist_df['w_temp'].max():.2f} °C")
print(f"Low Temp Count (<25C): {(wrist_df['w_temp'] < 25).sum()}")

# EDA Check (Skin conductance > 100uS is suspicious)
print(f"Max Chest EDA: {chest_df['c_eda'].max():.2f} uS")
print(f"Max Wrist EDA: {wrist_df['w_eda'].max():.2f} uS")

## 3.7 Sensor Flatline Check (Sensor Malfunction Indicator)

In [ ]:
def flatline_check(df, col, sid_col, freq_hz, min_duration_sec=10):
    threshold = int(freq_hz * min_duration_sec)
    print(
        f"\nFLATLINE CHECK: {col} (>{min_duration_sec}s at {freq_hz}Hz = {threshold} samples)"
    )
    found = False
    for sid in sorted(df[sid_col].unique()):
        vals = df[df[sid_col] == sid][col].values
        max_run = current_run = 1
        for i in range(1, len(vals)):
            if vals[i] == vals[i - 1]:
                current_run += 1
                max_run = max(max_run, current_run)
            else:
                current_run = 1
        if max_run > threshold:
            print(
                f"  S{sid}: max flatline = {max_run} samples ({max_run/freq_hz:.1f}s)"
            )
            found = True
    if not found:
        print("  No flatlines detected.")


# Chest sensors (700 Hz)
for col in ["ecg", "emg", "c_eda", "c_temp", "resp"]:
    flatline_check(chest_df, col, "sid", freq_hz=700)

# Wrist sensors
flatline_check(wrist_df, "w_eda", "sid", freq_hz=4)
flatline_check(wrist_df, "w_temp", "sid", freq_hz=4)
flatline_check(wrist_df, "bvp", "sid", freq_hz=64)

## 4: Distribution Analysis

### 4.1 Chest Sensors Histograms

In [ ]:
# Select sensor columns excluding sid and label
chest_sensors = [col for col in chest_df.columns if col not in ["sid", "label"]]

# Create histograms
fig = plt.figure(figsize=(16, 12))
gs = GridSpec(3, 3, figure=fig, hspace=0.4, wspace=0.3)

for idx, sensor in enumerate(chest_sensors):
    ax = fig.add_subplot(gs[idx // 3, idx % 3])

    # Plot histogram with KDE
    chest_df[sensor].hist(bins=50, alpha=0.7, edgecolor="black", ax=ax, density=True)
    chest_df[sensor].plot(kind="kde", ax=ax, color="red", linewidth=2)

    ax.set_title(f"{sensor.upper()}", fontsize=12, fontweight="bold")
    ax.set_xlabel("Value")
    ax.set_ylabel("Density")
    ax.grid(alpha=0.3)

plt.suptitle(
    "Chest Sensors - Distribution Analysis", fontsize=16, fontweight="bold", y=0.995
)
plt.savefig(FIGURES_PATH / "04_chest_distributions.png", dpi=300, bbox_inches="tight")
plt.show()

print(" Saved: 04_chest_distributions.png")

These observations are conclusive with lab conditions the experiments took place in. Accelerometer values are not widely spread and mostly concentrated around baseline zero (subjects were statiomary), heart electrical activity (ecg), respiration and muscle acivity emg can be similarly justified. Temperature has a narrow range around normal and expected values for humans and EDA shows the most variability which physiologically suggest the presence of different arousal levels

### 4.1.1 DISTRIBUTION BY LABEL

In [ ]:
print("Generating chest sensor distributions by label...")

# Sample for faster plotting
sample_size = 10000000
chest_sample = chest_usable.sample(min(sample_size, len(chest_usable)), random_state=42)

# Create figure
fig = plt.figure(figsize=(16, 12))
gs = GridSpec(3, 3, figure=fig, hspace=0.5, wspace=0.4)

chest_sensors = [col for col in chest_df.columns if col not in ["sid", "label"]]

for idx, sensor in enumerate(chest_sensors):
    ax = fig.add_subplot(gs[idx // 3, idx % 3])

    # Plot KDE for each label
    for label in USABLE_LABELS:
        data = chest_sample[chest_sample["label"] == label][sensor]
        if len(data) > 100:  # Only plot if enough data
            data.plot(
                kind="kde",
                ax=ax,
                label=LABEL_SHORT[label],
                color=LABEL_COLORS[label],
                linewidth=2.5,
                alpha=0.8,
            )

    ax.set_title(f"{sensor.upper()}", fontsize=12, fontweight="bold")
    ax.set_xlabel("Value", fontsize=10)
    ax.set_ylabel("Density", fontsize=10)
    ax.legend(loc="best", fontsize=9, framealpha=0.9)
    ax.grid(alpha=0.3)

plt.suptitle(
    "Chest Sensors - Distribution by Label\n(Baseline vs Stress vs Amusement)",
    fontsize=16,
    fontweight="bold",
    y=0.998,
)
plt.savefig(
    FIGURES_PATH / "04_chest_distributions_by_label.png", dpi=300, bbox_inches="tight"
)
plt.show()

print(" Saved: 04_chest_distributions_by_label.png")

### 4.2 Wrist Sensors Histograms

In [ ]:
wrist_sensors = [col for col in wrist_df.columns if col not in ["sid", "label"]]

fig = plt.figure(figsize=(16, 10))
gs = GridSpec(2, 4, figure=fig, hspace=0.4, wspace=0.3)

for idx, sensor in enumerate(wrist_sensors):
    ax = fig.add_subplot(gs[idx // 4, idx % 4])

    wrist_df[sensor].hist(bins=50, alpha=0.7, edgecolor="black", ax=ax, density=True)
    wrist_df[sensor].plot(kind="kde", ax=ax, color="red", linewidth=2)

    ax.set_title(f"{sensor.upper()}", fontsize=12, fontweight="bold")
    ax.set_xlabel("Value")
    ax.set_ylabel("Density")
    ax.grid(alpha=0.3)

plt.suptitle(
    "Wrist Sensors - Distribution Analysis", fontsize=16, fontweight="bold", y=0.995
)
plt.savefig(FIGURES_PATH / "04_wrist_distributions.png", dpi=300, bbox_inches="tight")
plt.show()

print(" Saved: 04_wrist_distributions.png")

### 4.2.1 DISTRIBUTION BY LABEL

In [ ]:
print("\nGenerating wrist sensor distributions by label...")

sample_size = 1000000

# Sample wrist data (commented out in the final version, did this to test first)
wrist_sample = wrist_usable.sample(min(sample_size, len(wrist_usable)), random_state=42)

# Create figure
fig = plt.figure(figsize=(16, 10))
gs = GridSpec(2, 4, figure=fig, hspace=0.5, wspace=0.4)

wrist_sensors = [col for col in wrist_df.columns if col not in ["sid", "label"]]

for idx, sensor in enumerate(wrist_sensors):
    ax = fig.add_subplot(gs[idx // 4, idx % 4])

    # Plot KDE for each label
    for label in USABLE_LABELS:
        data = wrist_sample[wrist_sample["label"] == label][sensor]
        if len(data) > 100:
            data.plot(
                kind="kde",
                ax=ax,
                label=LABEL_SHORT[label],
                color=LABEL_COLORS[label],
                linewidth=2.5,
                alpha=0.8,
            )

    ax.set_title(f"{sensor.upper()}", fontsize=12, fontweight="bold")
    ax.set_xlabel("Value", fontsize=10)
    ax.set_ylabel("Density", fontsize=10)
    ax.legend(loc="best", fontsize=9, framealpha=0.9)
    ax.grid(alpha=0.3)

plt.suptitle(
    "Wrist Sensors - Distribution by Label\n(Baseline vs Stress vs Amusement)",
    fontsize=16,
    fontweight="bold",
    y=0.998,
)
plt.savefig(
    FIGURES_PATH / "04_wrist_distributions_by_label.png", dpi=300, bbox_inches="tight"
)
plt.show()

print(" Saved: 04_wrist_distributions_by_label.png")

Sensors that show a good separation of patterns (easier to distinguish conditions) like both wrist and chest temp and eda have better predictive power for the model. For the ones that overlap, it might be necessary to do feature engineering and extract features with better separation.

### 4.3 Violin Plots by Chest Label

In [ ]:
# Select key sensors - essentially no accelerometers
key_chest_sensors = ["ecg", "c_eda", "c_temp", "resp"]

sample_size = 50000
chest_sample = chest_usable.sample(min(sample_size, len(chest_usable)), random_state=42)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for idx, sensor in enumerate(key_chest_sensors):
    ax = axes[idx]

    # Create violin plot
    parts = ax.violinplot(
        [
            chest_sample[chest_sample["label"] == label][sensor].values
            for label in USABLE_LABELS
        ],
        positions=range(len(USABLE_LABELS)),
        showmeans=True,
        showmedians=True,
        widths=0.7,
    )

    # Color violins
    for pc, label in zip(parts["bodies"], USABLE_LABELS):
        pc.set_facecolor(LABEL_COLORS[label])
        pc.set_alpha(0.7)

    ax.set_xticks(range(len(USABLE_LABELS)))
    ax.set_xticklabels([LABEL_SHORT[lbl] for lbl in USABLE_LABELS])
    ax.set_ylabel(f"{sensor.upper()} Value", fontsize=11)
    ax.set_title(
        f"{sensor.upper()} Distribution by Label", fontsize=12, fontweight="bold"
    )
    ax.grid(axis="y", alpha=0.3)

plt.suptitle(
    "Key Chest Sensors - Violin Plots by Label\n(Shape shows full distribution)",
    fontsize=14,
    fontweight="bold",
)
plt.tight_layout()
plt.savefig(FIGURES_PATH / "04_violin_plots_chest.png", dpi=300, bbox_inches="tight")
plt.show()

print(" Saved: 04_violin_plots_chest.png")

In [ ]:
# in the previous version of this there was no effect size included with the consequence that all values were considered statistically significant
# this led to the realization that as with chi-squared test, effect size also needs to be taken in mind
print("STATISTICAL SUMMARY BY LABEL")

key_sensors = ["ecg", "c_eda", "c_temp", "emg", "resp"]

# Helper function for Cohen's d interpretation


def interpret_cohens_d(d):
    """Interpret Cohen's d effect size"""
    d = abs(d)
    if d < 0.2:
        return "negligible"
    elif d < 0.5:
        return "small"
    elif d < 0.8:
        return "medium"
    else:
        return "large"


for sensor in key_sensors:
    print(f"\n{sensor.upper()}:")
    print("*" * 100)
    print(
        f"{'Label':<15} {'Mean':<12} {'Std':<12} {'Median':<12} {'Min':<12} {'Max':<12}"
    )
    print("*" * 100)

    # Collect data for each label
    data_by_label = {}
    for label in USABLE_LABELS:
        data = chest_usable[chest_usable["label"] == label][sensor]
        data_by_label[label] = data
        print(
            f"{LABEL_SHORT[label]:<15} "
            f"{data.mean():<12.3f} "
            f"{data.std():<12.3f} "
            f"{data.median():<12.3f} "
            f"{data.min():<12.3f} "
            f"{data.max():<12.3f}"
        )

    # Kruskal-Wallis test
    baseline_data = data_by_label[1]
    stress_data = data_by_label[2]
    amusement_data = data_by_label[3]

    h_stat, p_val = kruskal(baseline_data, stress_data, amusement_data)

    # EFFECT SIZE: Eta-squared (η²)  :  η² = (H - k + 1) / (n - k)
    # where H = Kruskal-Wallis statistic, n = total sample size , k = number of groups (3 here)

    n_total = len(baseline_data) + len(stress_data) + len(amusement_data)

    k = 3  # baseline, stress, amusement
    eta_squared = (h_stat - k + 1) / (n_total - k)

    #  Cohen's d for pairwise comparisons

    def cohens_d(group1, group2):
        mean1, mean2 = group1.mean(), group2.mean()
        std1, std2 = group1.std(), group2.std()
        n1, n2 = len(group1), len(group2)

        # Pooled standard deviation
        pooled_std = np.sqrt(((n1 - 1) * std1**2 + (n2 - 1) * std2**2) / (n1 + n2 - 2))

        # Cohen's d
        d = (mean1 - mean2) / pooled_std
        return d

    # Calculate Cohen's d for key comparisons
    d_baseline_stress = cohens_d(baseline_data, stress_data)
    d_baseline_amusement = cohens_d(baseline_data, amusement_data)
    d_stress_amusement = cohens_d(stress_data, amusement_data)

    print("STATISTICAL TEST RESULTS:\n")

    # P-value (statistical significance)
    print(f"\nKruskal-Wallis: H = {h_stat:.2f}, p = {p_val:.6f}")
    if p_val < 0.001:
        print("  Statistical significance:  p < 0.001")
    elif p_val < 0.05:
        print("  Statistical significance:  p < 0.05")
    else:
        print("  Statistical significance: Not significant (p ≥ 0.05)")

    # Effect size (practical significance)
    print(f"\nEffect Size (Eta-squared): η² = {eta_squared:.6f}")

    # Interpret eta-squared
    if eta_squared < 0.01:
        effect_interp = "Negligible effect"
    elif eta_squared < 0.06:
        effect_interp = "Small effect"
    elif eta_squared < 0.14:
        effect_interp = "Medium effect"
    else:
        effect_interp = "Large effect"

    print(f"  Interpretation: {effect_interp}")

    # Cohen's d for pairwise comparisons
    print("\nPairwise Effect Sizes (Cohen's d):")
    print(f"  Baseline vs Stress:     d = {d_baseline_stress:7.4f} ", end="")
    print(f"({interpret_cohens_d(abs(d_baseline_stress))})")

    print(f"  Baseline vs Amusement:  d = {d_baseline_amusement:7.4f} ", end="")
    print(f"({interpret_cohens_d(abs(d_baseline_amusement))})")

    print(f"  Stress vs Amusement:    d = {d_stress_amusement:7.4f} ", end="")
    print(f"({interpret_cohens_d(abs(d_stress_amusement))})")

    print("\n\nPractical Significance:")

    if p_val < 0.05 and eta_squared >= 0.01:
        print("    both statistically and practically significant")
        print("   → This sensor is likely useful for distinguishing labels")
    elif p_val < 0.05 and eta_squared < 0.01:
        print("    Statistically significant but negligible effect size")
        print("   → Differences are tiny and likely not useful")
        print("     due to large sample size detecting meaningless differences")
    else:
        print("    Not statistically significant")
        print("    This sensor does not differ between labels")

print(
    "\nEFFECT SIZE INTERPRETATION referenced from https://resources.nu.edu/statsresources/eta and https://resources.nu.edu/statsresources/cohensd"
)
print("\nEta-squared (η²):")
print("  < 0.01: Negligible")
print("  0.01-0.06: Small effect")
print("  0.06-0.14: Medium effect")
print("  > 0.14: Large effect")
print("\nCohen's d:")
print("  < 0.2: Negligible")
print("  0.2-0.5: Small")
print("  0.5-0.8: Medium")
print("  > 0.8: Large")

### 4.4 Box Plots by Label (Sample Sensor for simple demonstration)

In [ ]:
# bOX PLOTS (Sampled Data)

small_sample = 50000
chest_tiny = chest_usable.sample(min(small_sample, len(chest_usable)), random_state=42)

key_sensors = ["ecg", "c_eda", "c_temp", "resp"]

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for idx, sensor in enumerate(key_sensors):
    ax = axes[idx]

    # Prepare data for box plot
    data_by_label = [
        chest_tiny[chest_tiny["label"] == label][sensor].values
        for label in USABLE_LABELS
    ]

    # Create box plot
    bp = ax.boxplot(
        data_by_label,
        labels=[LABEL_SHORT[lbl] for lbl in USABLE_LABELS],
        patch_artist=True,
        showfliers=False,  # Hide outliers because of large sample size it will not be meaningful
        widths=0.6,
    )

    # Color boxes
    for patch, label in zip(bp["boxes"], USABLE_LABELS):
        patch.set_facecolor(LABEL_COLORS[label])
        patch.set_alpha(0.7)

    ax.set_ylabel(f"{sensor.upper()} Value", fontsize=11)
    ax.set_title(f"{sensor.upper()}", fontsize=12, fontweight="bold")
    ax.grid(axis="y", alpha=0.3)

plt.suptitle(
    "Chest Sensors by Label (Sampled, No Outliers)\n" + "Focus on median and quartiles",
    fontsize=14,
    fontweight="bold",
)
plt.tight_layout()
plt.savefig(FIGURES_PATH / "04_boxplots_clean.png", dpi=300, bbox_inches="tight")
plt.show()

print(" Saved: 04_boxplots_clean.png")

## 5: Outlier Detection

### 5.1 Z-Score Based Outlier Detection

In [ ]:
print("OUTLIER DETECTION (Z-Score > 3)")

# Calculate outliers for chest sensors
outlier_counts_chest = {}
for sensor in chest_sensors:
    z_scores = np.abs(zscore(chest_df[sensor]))
    outliers = (z_scores > 3).sum()
    outlier_counts_chest[sensor] = outliers

print("\nChest Sensors:")
for sensor, count in outlier_counts_chest.items():
    pct = (count / len(chest_df)) * 100
    print(f"  {sensor:15s}: {count:8,} outliers ({pct:.2f}%)")

# Calculate outliers for wrist sensors
outlier_counts_wrist = {}
for sensor in wrist_sensors:
    z_scores = np.abs(zscore(wrist_df[sensor]))
    outliers = (z_scores > 3).sum()
    outlier_counts_wrist[sensor] = outliers

print("\nWrist Sensors:")
for sensor, count in outlier_counts_wrist.items():
    pct = (count / len(wrist_df)) * 100
    print(f"  {sensor:15s}: {count:8,} outliers ({pct:.2f}%)")

### 5.2 IQR-Based Outlier Detection

In [ ]:
print("OUTLIER DETECTION with IQR Method")


def count_iqr_outliers(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return ((series < lower) | (series > upper)).sum()


print("\nChest Sensors:")
for sensor in chest_sensors:
    outliers = count_iqr_outliers(chest_df[sensor])
    pct = (outliers / len(chest_df)) * 100
    print(f"  {sensor:15s}: {outliers:8,} outliers ({pct:.2f}%)")

print("\nWrist Sensors:")
for sensor in wrist_sensors:
    outliers = count_iqr_outliers(wrist_df[sensor])
    pct = (outliers / len(wrist_df)) * 100
    print(f"  {sensor:15s}: {outliers:8,} outliers ({pct:.2f}%)")

Because this dataset concerns biosignals and outliers might indicate true rare physiology instead of being present solely due to sensor artifacts, pipeline errors or noise, the decision is to not directly trim or impute them , but apply channel-specific plausibility filters during later stages of preprocessing instead.

## 6: Correlation Analysis

### 6.1 Correlation Heatmap for Chest 

In [ ]:
print("Purpose: Detect multicollinearity and redundant features")
print("Method: Spearman (non-parametric, robust to non-normal distributions)")

print("\nCalculating chest sensor correlations...")

# Calculate Spearman correlation
chest_corr = chest_df[chest_sensors].corr(method="spearman")

# Plot heatmap
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    chest_corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    square=True,
    linewidths=1,
    cbar_kws={"shrink": 0.8},
    vmin=-1,
    vmax=1,
    ax=ax,
)
ax.set_title(
    "Chest Sensors - Spearman Correlation Matrix\n"
    + "(For Multicollinearity Detection)",
    fontsize=14,
    fontweight="bold",
)
plt.tight_layout()
plt.savefig(
    FIGURES_PATH / "06_chest_correlation_spearman.png", dpi=300, bbox_inches="tight"
)
plt.show()

print(" Saved: 06_chest_correlation_spearman.png")

### 6.2 Correlation Heatmap for Wrist 

In [ ]:
print("\nCalculating wrist sensor correlations...")

wrist_corr = wrist_df[wrist_sensors].corr(method="spearman")

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    wrist_corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    square=True,
    linewidths=1,
    cbar_kws={"shrink": 0.8},
    vmin=-1,
    vmax=1,
    ax=ax,
)
ax.set_title(
    "Wrist Sensors - Spearman Correlation Matrix\n"
    + "(For Multicollinearity Detection)",
    fontsize=14,
    fontweight="bold",
)
plt.tight_layout()
plt.savefig(
    FIGURES_PATH / "06_wrist_correlation_spearman.png", dpi=300, bbox_inches="tight"
)
plt.show()

print(" Saved: 06_wrist_correlation_spearman.png")

The vast majority of signals appear to be independent (|r| < 3). The only highly correlated signals are the ones relating to different motion axes for the chest sensors, which is an expected result

A global feature-to-label correlation analysis was omitted due to inter-subject variability which is inherent in biosignals. An individual's baseline state can differ significantly from others, for example, one subject's resting EDA levels might exceed another subject's stressed levels. Consequently, aggregating data across all subjects for a global correlation analysis tends to 'wash out' the predictive power of features that are highly effective within an individual context. 

### 6.3 Pairplot (Sample Sensors)

In [ ]:
# Select subset of sensors for computational efficiency
sample_sensors = ["ecg", "emg", "c_eda", "c_temp", "resp"]
sample_data = chest_usable[sample_sensors + ["label"]].sample(100000, random_state=42)
sample_data["label"] = sample_data["label"].map(LABEL_SHORT)

# Create pairplot
pairplot = sns.pairplot(
    sample_data,
    hue="label",
    palette=STRING_LABEL_COLORS,
    diag_kind="kde",
    plot_kws={"alpha": 0.6, "s": 10},
    height=2.5,
)
pairplot.fig.suptitle(
    "Chest Sensors Pairplot (100K sample)", fontsize=14, fontweight="bold", y=1.001
)
plt.savefig(FIGURES_PATH / "06_pairplot_sample.png", dpi=300, bbox_inches="tight")
plt.show()

print(" Saved: 06_pairplot_sample.png")

## 7: Time Series Visualization

### 7.1 Plot Signals for Sample Subject (S2)

In [ ]:
s2_chest = chest_df[chest_df["sid"] == 2].reset_index(drop=True)

# Plot first 60 seconds
sample_length = SF_CHEST * 60
s2_sample = s2_chest.iloc[:sample_length]

fig, axes = plt.subplots(len(chest_sensors), 1, figsize=(16, 12), sharex=True)
time = np.arange(len(s2_sample)) / SF_CHEST  # Time in seconds

for idx, sensor in enumerate(chest_sensors):
    axes[idx].plot(time, s2_sample[sensor], linewidth=0.5, color="steelblue")
    axes[idx].set_ylabel(sensor, fontsize=10, fontweight="bold")
    axes[idx].grid(alpha=0.3)

axes[-1].set_xlabel("Time (seconds)", fontsize=12)
plt.suptitle(
    "Subject 2 - First 60 Seconds (Chest Sensors)", fontsize=14, fontweight="bold"
)
plt.tight_layout()
plt.savefig(FIGURES_PATH / "07_timeseries_s2.png", dpi=300, bbox_inches="tight")
plt.show()

print("Saved: 07_timeseries_s2.png")

### 7.2 Sample Signal Colored by Label

In [ ]:
# Plot EDA with labels colored
fig, ax = plt.subplots(figsize=(16, 5))

# Sample  roughly 1.5 hours
sample_len = SF_CHEST * 6000
s2_minute = s2_chest.iloc[:sample_len]
time = np.arange(len(s2_minute)) / SF_CHEST

# Plot signal
for label in USABLE_LABELS:
    mask = s2_minute["label"] == label
    if mask.any():
        ax.scatter(
            time[mask],
            s2_minute.loc[mask, "c_eda"],
            c=LABEL_COLORS[label],
            label=LABEL_SHORT[label],
            s=1,
            alpha=0.7,
        )

ax.set_xlabel("Time (seconds)", fontsize=12)
ax.set_ylabel("EDA", fontsize=12)
ax.set_title("Subject 2 - EDA Colored by Label", fontsize=14, fontweight="bold")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_PATH / "07_eda_by_label.png", dpi=300, bbox_inches="tight")
plt.show()

print(" Saved: 07_eda_by_label.png")

## 8: Per-Subject Analysis

### 8.1 Statistical Summary per Subject

In [ ]:
# Calculate stats per subject for a key sensor (eg chest EDA)
subject_stats = []

for sid in SUBJECT_IDS:
    subj_data = chest_df[chest_df["sid"] == sid]["c_eda"]
    stats_dict = {
        "Subject": f"S{sid}",
        "Count": len(subj_data),
        "Mean": subj_data.mean(),
        "Std": subj_data.std(),
        "Min": subj_data.min(),
        "Max": subj_data.max(),
        "Skew": skew(subj_data),
        "Kurtosis": kurtosis(subj_data),
    }
    subject_stats.append(stats_dict)

stats_df = pd.DataFrame(subject_stats)

print("\nPER-SUBJECT STATISTICS (Chest EDA)")

display(stats_df)

In [ ]:
# wrist eda
subject_stats = []

for sid in SUBJECT_IDS:
    subj_data = wrist_df[wrist_df["sid"] == sid]["w_eda"]
    stats_dict = {
        "Subject": f"S{sid}",
        "Count": len(subj_data),
        "Mean": subj_data.mean(),
        "Std": subj_data.std(),
        "Min": subj_data.min(),
        "Max": subj_data.max(),
        "Skew": skew(subj_data),
        "Kurtosis": kurtosis(subj_data),
    }
    subject_stats.append(stats_dict)

stats_df = pd.DataFrame(subject_stats)

print("\nPER-SUBJECT STATISTICS (wrist EDA)")

display(stats_df)

# 9. Examining subject non heterogeneity

The goal is to prove that subjects are significantly different from each other in their physiological baselines and responses. This justifies:

- Personalized models (one-size-fits-all won't work)
- Federated learning (can't just pool all data)
- Leave-One-Subject-Out CV (subjects are independent units)

## 9.1: Test if subject means differ

In [ ]:
from scipy.stats import f_oneway, kruskal

print("BASELINE HETEROGENEITY TEST")
print("Question: Do subjects have different baseline EDA levels?")
print("Method: One-way ANOVA (Kruskal-Wallis if non-normal)")

# Get baseline data per subject
baseline_data_by_subject = []
for sid in SUBJECT_IDS:
    # Get only BASELINE label (label == 1)
    baseline_subj = chest_df[(chest_df["sid"] == sid) & (chest_df["label"] == 1)][
        "c_eda"
    ]
    baseline_data_by_subject.append(baseline_subj.values)

# Test normality (Shapiro-Wilk on sample)
sample_for_normality = np.concatenate([s[:1000] for s in baseline_data_by_subject])
_, p_normality = stats.shapiro(sample_for_normality[:5000])  # Sample for speed

print(f"\nNormality test p-value: {p_normality:.6f}")

if p_normality < 0.05:
    print("→ Data is not normally distributed, using Kruskal-Wallis")
    test_name = "Kruskal-Wallis"
    h_stat, p_value = kruskal(*baseline_data_by_subject)
    test_stat_name = "H"
    test_stat = h_stat
else:
    print("→ Data is normally distributed, using One-way ANOVA")
    test_name = "ANOVA"
    f_stat, p_value = f_oneway(*baseline_data_by_subject)
    test_stat_name = "F"
    test_stat = f_stat

print(f"\n{test_name} Results:")
print(f"  {test_stat_name}-statistic: {test_stat:.2f}")
print(f"  P-value: {p_value:.10f}")

if p_value < 0.001:
    print("  → HIGHLY SIGNIFICANT (p < 0.001)")
    print("  → Subjects have DIFFERENT baseline EDA levels")
else:
    print("  → Not significant")

## 9.2: Quantify effect size (how much means differ practically)   

In [ ]:
print("EFFECT SIZE ANALYSIS")
print("With large N, everything is 'significant', therefore quantifying effect size")

# Calculate means per subject
subject_means = [np.mean(data) for data in baseline_data_by_subject]
subject_stds = [np.std(data) for data in baseline_data_by_subject]

# effect Size 1: Coefficient of Variation (CV)
cv_between_subjects = (np.std(subject_means) / np.mean(subject_means)) * 100

print("\n1. COEFFICIENT OF VARIATION (Between Subjects):")
print(f"   CV = {cv_between_subjects:.2f}%")
print("\n   Interpretation:")
if cv_between_subjects < 15:
    print(" Low heterogeneity (subjects are similar)")
elif cv_between_subjects < 30:
    print(" Moderate heterogeneity")
elif cv_between_subjects < 50:
    print(" High heterogeneity")
else:
    print(" Very high heterogeneity")

# Effect Size 2: Intraclass Correlation Coefficient (ICC)

grand_mean = np.mean(subject_means)
ss_between = sum(
    len(data) * (mean - grand_mean) ** 2
    for data, mean in zip(baseline_data_by_subject, subject_means)
)
ss_within = sum(
    np.sum((data - mean) ** 2)
    for data, mean in zip(baseline_data_by_subject, subject_means)
)

n_subjects = len(SUBJECT_IDS)
n_total = sum(len(data) for data in baseline_data_by_subject)
ms_between = ss_between / (n_subjects - 1)
ms_within = ss_within / (n_total - n_subjects)

icc = (ms_between - ms_within) / (
    ms_between + (len(baseline_data_by_subject[0]) - 1) * ms_within
)

print("\n2. INTRACLASS CORRELATION COEFFICIENT (ICC):")
print(f"   ICC = {icc:.4f}")
print("\n   Interpretation:")
if icc < 0.5:
    print(" Poor reliability (< 0.5)")
    print(" Subjects are quite different from each other")
elif icc < 0.75:
    print(" Moderate reliability (0.5-0.75)")
    print(" Some between-subject variance")
elif icc < 0.9:
    print(" Good reliability (0.75-0.9)")
    print(" Significant between-subject variance")
else:
    print(" Excellent reliability (> 0.9)")
    print(" High between-subject variance")

# === Effect Size 3: Eta-squared (η²) ===
eta_squared = ss_between / (ss_between + ss_within)

print("\n3. ETA-SQUARED (η²) - Proportion of variance between subjects:")
print(f"   η² = {eta_squared:.4f} ({eta_squared*100:.2f}% of variance)")
print("\n   Interpretation:")
if eta_squared < 0.06:
    print(" Small effect")
elif eta_squared < 0.14:
    print(" Medium effect")
else:
    print(" Large effect")

# === Effect Size 4: Range Ratio ===
min_mean = min(subject_means)
max_mean = max(subject_means)
range_ratio = max_mean / min_mean if min_mean > 0 else np.inf

print("\n4. RANGE RATIO (Max/Min baseline):")
print(f"   Lowest subject mean: {min_mean:.4f}")
print(f"   Highest subject mean: {max_mean:.4f}")
print(f"   Ratio: {range_ratio:.2f}x")
print("\n   Interpretation:")
if range_ratio < 1.5:
    print(" Subjects are similar (< 1.5x difference)")
elif range_ratio < 2.0:
    print(" Moderate differences (1.5-2x)")
elif range_ratio < 3.0:
    print(" Large differences (2-3x)")
else:
    print(" Very large differences (> 3x)")

Interpretation:
- We observe ubjects have very different CVs. This means their signals have different stability/volatility, because even if two subjects have the same mean, one can have much more fluctuation.
- ICC decomposes variance to between-subject variance (if high, indicates subjects have different baselines), and total variance (since each subject also fluctuates around their own baseline). Our observation f high ICC means that measurements from the same subject are more similar to each other than to measurements from other subjects, which indicates subject-dependent data heterogeneity.
- η² answers the question of “How much of the total variability in a feature is explained by group membership?” Since η² is large after grouping by subject, we can interpret this to mean that subject identity explains a lot of signal variability, therefore the dataset is heterogeneous
- Range Ratio examines how far apart are subjects’ typical values, compared to the typical variation (spread) within a subject. Large ratio indicates subjects differ a lot relative to within-subject fluctuations, therefore the nature of the dataset is heterogeneous.



## 9.3: Test if subjects respond differently to stress 

In [ ]:
print("STRESS RESPONSE HETEROGENEITY")

# Calculate stress reactivity per subject (mean of Stress EDA - mean of Baseline EDA) and store the deltas to compare the,m
stress_reactivity = []
subject_ids_for_reactivity = []

for sid in SUBJECT_IDS:
    baseline_mean = chest_df[(chest_df["sid"] == sid) & (chest_df["label"] == 1)][
        "c_eda"
    ].mean()
    stress_mean = chest_df[(chest_df["sid"] == sid) & (chest_df["label"] == 2)][
        "c_eda"
    ].mean()

    reactivity = stress_mean - baseline_mean
    stress_reactivity.append(reactivity)
    subject_ids_for_reactivity.append(sid)

# Test if reactivity varies significantly between subjects
# Use CV and range of reactivity values

cv_reactivity = (np.std(stress_reactivity) / np.abs(np.mean(stress_reactivity))) * 100
min_reactivity = min(stress_reactivity)
max_reactivity = max(stress_reactivity)

print("\nSTRESS REACTIVITY ANALYSIS:")
print(f"  Mean reactivity (Stress - Baseline): {np.mean(stress_reactivity):.4f}")
print(f"  Std of reactivity: {np.std(stress_reactivity):.4f}")
print(f"  CV of reactivity: {cv_reactivity:.2f}%")
print(f"  Range: {min_reactivity:.4f} to {max_reactivity:.4f}")

if cv_reactivity > 50:
    print("\n  HIGHLY HETEROGENEOUS STRESS RESPONSES")
    print("  Different subjects respond very differently to stress")
elif cv_reactivity > 30:
    print("\n  MODERATELY HETEROGENEOUS STRESS RESPONSES")
    print("  Subjects respond differently to stress")
else:
    print("\n  Subjects respond similarly to stress")

# Identify subjects with unusual responses
mean_reactivity = np.mean(stress_reactivity)
std_reactivity = np.std(stress_reactivity)

outlier_subjects = []
for sid, react in zip(subject_ids_for_reactivity, stress_reactivity):
    z_score = (react - mean_reactivity) / std_reactivity
    if abs(z_score) > 1.5:
        outlier_subjects.append((sid, react, z_score))

if outlier_subjects:
    print("\n  Subjects with unusual stress responses (|z| > 1.5):")
    for sid, react, z in outlier_subjects:
        print(f"    S{sid}: Reactivity = {react:.4f} (z = {z:+.2f})")

# VISUALIZATION: Subject Heterogeneity

print("GENERATING VISUALIZATIONS")

fig = plt.figure(figsize=(18, 12))
gs = fig.add_gridspec(3, 3, hspace=0.4, wspace=0.3)

# Plot 1: Baseline EDA by subject
ax1 = fig.add_subplot(gs[0, :2])
subjects_labels = [f"S{sid}" for sid in SUBJECT_IDS]
ax1.bar(subjects_labels, subject_means, color="steelblue", alpha=0.7, edgecolor="black")
ax1.axhline(
    np.mean(subject_means),
    color="red",
    linestyle="--",
    linewidth=2,
    label="Overall Mean",
)
ax1.set_xlabel("Subject", fontsize=12)
ax1.set_ylabel("Mean Baseline EDA", fontsize=12)
ax1.set_title(
    "Baseline EDA Heterogeneity Across Subjects", fontsize=14, fontweight="bold"
)
ax1.legend()
ax1.grid(axis="y", alpha=0.3)
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45)

# Plot 2: CV text box
ax2 = fig.add_subplot(gs[0, 2])
ax2.axis("off")
textstr = (
    f"HETEROGENEITY METRICS:\n\n"
    f"CV: {cv_between_subjects:.1f}%\n"
    f"η²: {eta_squared:.3f}\n"
    f"ICC: {icc:.3f}\n"
    f"Range: {range_ratio:.2f}x\n\n"
    f"P-value: {p_value:.2e}"
)
ax2.text(
    0.1,
    0.5,
    textstr,
    fontsize=12,
    verticalalignment="center",
    bbox={"boxstyle": "round", "facecolor": "wheat", "alpha": 0.5},
)

# Plot 3: Distribution of baseline values per subject (violin plot)
ax3 = fig.add_subplot(gs[1, :])
violin_data = []
violin_labels = []
for sid in SUBJECT_IDS:
    data = (
        chest_df[(chest_df["sid"] == sid) & (chest_df["label"] == 1)]["c_eda"]
        .sample(
            min(
                1000, len(chest_df[(chest_df["sid"] == sid) & (chest_df["label"] == 1)])
            ),
            random_state=42,
        )
        .values
    )
    violin_data.append(data)
    violin_labels.append(f"S{sid}")

parts = ax3.violinplot(
    violin_data,
    positions=range(len(SUBJECT_IDS)),
    showmeans=True,
    showmedians=True,
    widths=0.7,
)
ax3.set_xticks(range(len(SUBJECT_IDS)))
ax3.set_xticklabels(violin_labels, rotation=45)
ax3.set_xlabel("Subject", fontsize=12)
ax3.set_ylabel("Baseline EDA", fontsize=12)
ax3.set_title(
    "Distribution of Baseline EDA per Subject\n(Wide shapes = high within-subject variability)",
    fontsize=13,
    fontweight="bold",
)
ax3.grid(axis="y", alpha=0.3)

# Plot 4: Stress reactivity per subject
ax4 = fig.add_subplot(gs[2, :2])
colors = ["green" if r > mean_reactivity else "coral" for r in stress_reactivity]
bars = ax4.bar(
    subjects_labels, stress_reactivity, color=colors, alpha=0.7, edgecolor="black"
)
ax4.axhline(
    mean_reactivity, color="red", linestyle="--", linewidth=2, label="Mean Reactivity"
)
ax4.axhline(0, color="black", linestyle="-", linewidth=1)
ax4.set_xlabel("Subject", fontsize=12)
ax4.set_ylabel("Stress Reactivity (Δ EDA)", fontsize=12)
ax4.set_title(
    "Stress Response Heterogeneity\n(Green = Above average, Coral = Below average)",
    fontsize=13,
    fontweight="bold",
)
ax4.legend()
ax4.grid(axis="y", alpha=0.3)
plt.setp(ax4.xaxis.get_majorticklabels(), rotation=45)

# Plot 5: Reactivity CV text
ax5 = fig.add_subplot(gs[2, 2])
ax5.axis("off")
textstr2 = (
    f"STRESS RESPONSE:\n\n"
    f"Mean Δ: {np.mean(stress_reactivity):.3f}\n"
    f"Std Δ: {np.std(stress_reactivity):.3f}\n"
    f"CV: {cv_reactivity:.1f}%\n"
    f"Range: {min_reactivity:.3f}\n"
    f"       to {max_reactivity:.3f}"
)
ax5.text(
    0.1,
    0.5,
    textstr2,
    fontsize=12,
    verticalalignment="center",
    bbox={"boxstyle": "round", "facecolor": "lightblue", "alpha": 0.5},
)

plt.suptitle(
    "Inter-Subject Heterogeneity Analysis: Evidence for Personalized FL",
    fontsize=16,
    fontweight="bold",
    y=0.995,
)
plt.savefig(FIGURES_PATH / "09_subject_heterogeneity.png", dpi=300, bbox_inches="tight")
plt.show()

print("Saved: 09_subject_heterogeneity.png")

Key findings:
- Subjects have significantly different baseline characteristics
- Effect sizes indicate practical importance (not just statistical)
- Subjects respond differently to stress

Conclusion: subject heterogeneity analysis revealed significant differences in baseline EDA (CV = 73.7%, η² = 0.976, p < 0.001) and stress reactivity (CV = 89.2%), providing strong
   empirical justification for personalized federated learning approaches rather than one-size-fits-all global models.

## NOTE TO SELF: DO NOT FORGET TO RUN PCA AND TSNE AFTER EXTRACTING TJE FEATURES!!! DOING IT ON THE RAW SIGNAL IN THE TIME SERIES IS BOTH GOING TO COMPUTATIONALLY EXPLODE AD MEANINGLESS (WE WILL SEE NEIGHBORS WITH SIMILAR VALUES EG IN EDA BUT ONE IS FROM THE BASELINE AND THE OTHER FROM STRESS), SO: Phase 1: Clean & Synchronize (Done), Phase 2: Sliding Windows (Next Step) Phase 3: Feature Extraction (Calculate Mean, Std, Peaks, etc.) Phase 4: --> pca T-SNE HERE <-- (On the features)

# I CAN USE TSNE AGAIN IN THE FUTURE! Inspect separability of classes using features (from NeuroKit2, TSFEL, etc.), See if different subjects cluster separately (subject bias), Check whether adding certain features (e.g. HRV or EDA) makes clusters cleaner.


# AFTER HIST AND BOX PLOT DISTS FOR THE WHOLE MERGED GROUP ARE DONE I CAN GROUP THEM BY SUBJECT AND DO A VARIATION ANALYSIS AGAIN. HOPEFULLY I WILL GET STATISTICALLY SIGNIFICANT RESULTS THAT REINFORCE THE ARGUMENT FOR FL AND PFL

# NOTE FOR TOMORROW - STOP HERRE!!! WHAT SHOULD I DO WITH THOSE 11 NEGATIVES, PERHAPS INTERPOLATE AS WELL?? INVESTIGATE